# ORIE 6365 Practical Assignment Experiments

## Imports & Setup

In [26]:
%reload_ext autoreload
%autoreload 2

from __future__ import annotations

import math
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm

from data import generate_data
import grad_methods
from loss import QuadraticLoss, LogisticLoss


TITLE_FONTSIZE = 26
LABEL_FONTSIZE = 20
TICK_FONTSIZE = 14
LEGEND_FONTSIZE = 16

plt.rcParams.update({
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": LABEL_FONTSIZE,
    "axes.titlesize": TITLE_FONTSIZE,
    "figure.titlesize": TITLE_FONTSIZE,
    "axes.labelsize": LABEL_FONTSIZE,
    "xtick.labelsize": TICK_FONTSIZE,
    "ytick.labelsize": TICK_FONTSIZE,
    "legend.fontsize": LEGEND_FONTSIZE,
})

## Configuration

In [33]:
SEED = 6365
SAVE_FIGURES = True
FIGURE_DIR = Path("figures")
FIGURE_DIR.mkdir(exist_ok=True)

SIZES = [(100, 100), (100, 1000)]
SIGMAS = [1e1, 1e3, 1e5]
MUS = [0, 1e-3, 1e-2]
SMOOTH_LOSSES = ["quadratic", "logistic"]
ADAPTIVE_OPTIONS = [False, True]
SMOOTH_N_ITERS = 500

SUBGRADIENT_LOSSES = ["quadratic", "logistic", "l1"]
R_VALUES = [1.0, 10.0]
SUBGRADIENT_ITERS = [500, 2000]
NORMALIZED_OPTIONS = [False, True]

DEFAULT_N, DEFAULT_M = (100, 100)
DEFAULT_SIGMA = 1e3
DEFAULT_MU = 0
DEFAULT_R = 1.0
DEFAULT_K = 2000

rng = np.random.default_rng(SEED)
SMOOTH_X0 = {n: rng.normal(size=n) for n in sorted({n for n, _ in SIZES})}

## Helpers

In [27]:
SMOOTH_METHODS = {
    "gradient_method": grad_methods.gradient_method,
    "fast_gradient_method": grad_methods.fast_gradient_method,
}
LOSS_CLASSES = {"quadratic": QuadraticLoss, "logistic": LogisticLoss}
METHOD_LABEL = {"gradient_method": "Gradient", "fast_gradient_method": "Fast Gradient"}
METHOD_LINESTYLE = {"gradient_method": "-", "fast_gradient_method": "--"}
X_LABEL = {"iteration": "Iteration k", "time": "Time (s)", "mat_vec": "Matrix-Vector Operations"}
Y_LABEL = {"func_res": r"$f(x_k)-f^\star$", "grad": r"$\|\nabla f(x_k)\|_2$"}

def keep(rows: list[dict[str, Any]], **filters: Any) -> list[dict[str, Any]]:
    out = []
    for row in rows:
        ok = True
        for key, value in filters.items():
            actual = row.get(key)
            if isinstance(value, (list, tuple, set)):
                ok = any(keep([row], **{key: option}) for option in value)
            elif isinstance(actual, float) or isinstance(value, float):
                ok = np.isclose(float(actual), float(value))
            else:
                ok = actual == value
            if not ok:
                break
        if ok:
            out.append(row)
    return out


def add_function_residuals_using_best_observed_value(rows: list[dict[str, Any]], group_keys: tuple[str, ...]) -> None:
    """Add history['func_res'] using the best objective value observed within each group.
        We also add history['iteration'], for use in plotting.
    """
    groups = {}
    for row in rows:
        groups.setdefault(tuple(row[k] for k in group_keys), []).append(row)

    for group in groups.values():
        f_star = min(float(np.min(row["history"]["func"])) for row in group)
        for row in group:
            history = row["history"]
            history["func_res"] = np.asarray(history["func"], dtype=float) - f_star
            history["iteration"] = np.arange(len(history["func"]))


def savefig(fig: plt.Figure, name: str) -> None:
    if SAVE_FIGURES:
        fig.savefig(FIGURE_DIR / f"{name}.pdf", bbox_inches="tight")
        # fig.savefig(FIGURE_DIR / f"{name}.png", dpi=200, bbox_inches="tight")


## Data Check

In [11]:
print("Generated data condition numbers")
print("n     m     target sigma    observed cond    relative error")

for n, m in SIZES:
    for sigma in SIGMAS:
        A, b = generate_data(n=n, m=m, sigma=sigma, seed=SEED)
        eigvals = np.linalg.eigvalsh(A.T @ A)
        cond = float(eigvals.max() / eigvals.min())
        rel_error = abs(cond - sigma) / sigma
        print(f"{n:<5} {m:<5} {sigma:<14.3g} {cond:<16.6g} {rel_error:.3e}")


Generated data condition numbers
n     m     target sigma    observed cond    relative error
100   100   10             10               3.553e-15
100   100   1e+03          1000             1.697e-13
100   100   1e+05          100000           1.274e-12
100   1000  10             10               9.237e-15
100   1000  1e+03          1000             1.035e-14
100   1000  1e+05          100000           7.448e-12


## Smooth Sweeps

In [12]:
smooth_results = []
total = len(SIZES) * len(SIGMAS) * len(MUS) * len(SMOOTH_LOSSES) * len(ADAPTIVE_OPTIONS) * len(SMOOTH_METHODS)

with tqdm(total=total, desc="Smooth") as pbar:
    for n, m in SIZES:
        x0 = SMOOTH_X0[n]
        for sigma in SIGMAS:
            A, b = generate_data(n=n, m=m, sigma=sigma, seed=SEED)
            for mu in MUS:
                for loss in SMOOTH_LOSSES:
                    L0 = LOSS_CLASSES[loss].lipschitz_estimate(A, mu)
                    for adaptive in ADAPTIVE_OPTIONS:
                        for method_name, method in SMOOTH_METHODS.items():
                            pbar.set_postfix_str(f"{loss}, sigma={sigma:g}, mu={mu:g}, adaptive={adaptive}, {method_name}")
                            _, history, status = method(A, b, loss, mu, x0, SMOOTH_N_ITERS, L0, adaptive=adaptive, show_progress=False)
                            smooth_results.append({
                                "n": n, "m": m, "sigma": float(sigma), "mu": float(mu), "loss": loss,
                                "adaptive": adaptive, "method": method_name, "n_iters": SMOOTH_N_ITERS,
                                "L0": float(L0), "status": status, "history": history,
                            })
                            pbar.update(1)

add_function_residuals_using_best_observed_value(
    smooth_results,
    ("n", "m", "sigma", "mu", "loss")
)
len(smooth_results)

Smooth:   0%|          | 0/144 [00:00<?, ?it/s]

144

## Smooth Gradient Plots

In [35]:
def plot_smooth_parameter_sweep(
        loss: str, sweep: str, n=DEFAULT_N, m=DEFAULT_M, adaptive: bool = False, mu=DEFAULT_MU, sigma=DEFAULT_SIGMA
) -> plt.Figure:
    fixed = {"sigma": "mu", "mu": "sigma"}[sweep]
    if sweep == "sigma":
        rows = keep(smooth_results, n=n, m=m, loss=loss, mu=mu, adaptive=adaptive)
        sweep_values = SIGMAS
        fixed_display = f"$\\mu$={mu:g}"
    elif sweep == "mu":
        rows = keep(smooth_results, n=n, m=m, loss=loss, sigma=sigma, adaptive=adaptive)
        sweep_values = MUS
        fixed_display = f"$\\sigma$={sigma:g}"
    else:
        raise ValueError(sweep)

    fig, axes = plt.subplots(2, 3, figsize=(18, 9), sharex="col", sharey="row", constrained_layout=True)
    for i, y in enumerate(["func_res", "grad"]):
        for j, x in enumerate(["iteration", "time", "mat_vec"]):
            for value in sweep_values:
                ax = axes[i, j]
                for method in SMOOTH_METHODS:            
                    row = keep(rows, method=method, **{sweep: value})[0]
                    ax.plot(row["history"][x], row["history"][y], label=f"$\\{sweep}$={value:g}", 
                            color=f"C{sweep_values.index(value)}", linestyle=METHOD_LINESTYLE[method], linewidth=2)
                ax.set_yscale("log")
                if j == 0:
                    ax.set_ylabel(Y_LABEL[y])
                if i == 1:
                    ax.set_xlabel(X_LABEL[x])

        # Add custom legend entries for sweep values and methods
        sweep_handles = [plt.Line2D([0], [0], color=f"C{i}", linestyle="-", linewidth=2) 
                         for i in range(len(sweep_values))]
        sweep_labels = [f"$\\{sweep}$={v:g}" for v in sweep_values]
        method_handles = [plt.Line2D([0], [0], color="black", linestyle=METHOD_LINESTYLE[m], linewidth=2) 
                          for m in SMOOTH_METHODS.keys()]
        method_labels = [METHOD_LABEL[m] for m in SMOOTH_METHODS.keys()]
        for j in range(3):
            axes[i, j].legend(handles=sweep_handles + method_handles, 
                            labels=sweep_labels + method_labels, loc="best")
    fig.suptitle(f"{loss.title()} loss - {fixed_display}, n={n}, m={m}, adaptive={adaptive}")
    savefig(fig, f"smooth_sweep-{sweep}_{loss}_n-{n}_m-{m}_{fixed}-{eval(fixed)}_adaptive-{adaptive}")
    return fig

for loss in SMOOTH_LOSSES:
    for n, m in SIZES:
        for adaptive in ADAPTIVE_OPTIONS:
            for mu in MUS:
                fig = plot_smooth_parameter_sweep(loss, "sigma", n=n, m=m, adaptive=adaptive, mu=mu)
                plt.close()  # plt.show()
            for sigma in SIGMAS:
                fig = plot_smooth_parameter_sweep(loss, "mu", n=n, m=m, adaptive=adaptive, sigma=sigma)
                plt.close()
                # plt.show()

## Subgradient Sweeps

In [ ]:
def subgradient_gamma(A: np.ndarray, b: np.ndarray, loss: str, R: float, K: int, normalized: bool) -> float:
    gamma = R / math.sqrt(K)
    if not normalized:
        gamma /= max(grad_methods.subgrad_norm_bound(A, b, loss, R), 1e-12)
    return gamma

subgradient_results = []
total = len(SIZES) * len(SIGMAS) * len(SUBGRADIENT_LOSSES) * len(R_VALUES) * len(SUBGRADIENT_ITERS) * len(NORMALIZED_OPTIONS)

with tqdm(total=total, desc="Subgradient") as pbar:
    for n, m in SIZES:
        x0 = np.zeros(n)
        for sigma in SIGMAS:
            A, b = generate_data(n=n, m=m, sigma=sigma, seed=SEED)
            for loss in SUBGRADIENT_LOSSES:
                for R in R_VALUES:
                    for K in SUBGRADIENT_ITERS:
                        for normalized in NORMALIZED_OPTIONS:
                            gamma = subgradient_gamma(A, b, loss, R, K, normalized)
                            pbar.set_postfix_str(f"{loss}, sigma={sigma:g}, R={R:g}, K={K}, normalized={normalized}")
                            _, history, status = grad_methods.subgradient_method(A, b, loss, R, x0, K, gamma, normalized=normalized, show_progress=False)
                            subgradient_results.append({
                                "n": n, "m": m, "sigma": float(sigma), "loss": loss, "R": float(R),
                                "n_iters": K, "gamma": gamma, "normalized": normalized,
                                "method": "subgradient_method", "status": status, "history": history,
                            })
                            pbar.update(1)

add_function_residuals_using_best_observed_value(subgradient_results, ("n", "m", "sigma", "loss", "R", "n_iters"))
len(subgradient_results)

## Subgradient Plots

In [ ]:
def plot_subgradient(loss: str) -> plt.Figure:
    rows = sorted(
        keep(subgradient_results, n=DEFAULT_N, m=DEFAULT_M, sigma=DEFAULT_SIGMA, loss=loss, R=DEFAULT_R, n_iters=DEFAULT_K),
        key=lambda r: r["normalized"],
    )
    fig, axes = plt.subplots(2, 3, figsize=(13, 6), constrained_layout=True)
    x_defs = [
        (lambda h: np.arange(len(h["func_res"])), "Iteration k"),
        (lambda h: h["time"], "Time (s)"),
        (lambda h: h["mat_vec"], "Matrix-vector products"),
    ]
    for row in rows:
        h = row["history"]
        label = "Normalized" if row["normalized"] else "Unnormalized"
        for j, (xfn, xlabel) in enumerate(x_defs):
            axes[0, j].plot(xfn(h), h["func_res"], label=label)
            axes[1, j].plot(xfn(h), h["grad"], label=label)
            axes[0, j].set_yscale("log")
            axes[1, j].set_yscale("log")
            axes[0, j].set_xlabel(xlabel)
            axes[1, j].set_xlabel(xlabel)
            axes[0, j].set_ylabel(r"$f(x_k)-f^\star$")
            axes[1, j].set_ylabel("Subgradient norm")
    for ax in axes.ravel():
        ax.legend()
    fig.suptitle(f"Subgradient {loss}, sigma={DEFAULT_SIGMA:g}, R={DEFAULT_R:g}, K={DEFAULT_K}")
    savefig(fig, f"subgradient_{loss}")
    return fig

for loss in SUBGRADIENT_LOSSES:
    fig = plot_subgradient(loss)
    plt.show()

In [ ]:
def plot_subgradient_parameter_sweep(loss: str, sweep: str) -> plt.Figure:
    rows = keep(subgradient_results, n=DEFAULT_N, m=DEFAULT_M, sigma=DEFAULT_SIGMA, loss=loss, normalized=True)
    if sweep == "R":
        rows = keep(rows, n_iters=DEFAULT_K)
        values = R_VALUES
    elif sweep == "n_iters":
        rows = keep(rows, R=DEFAULT_R)
        values = SUBGRADIENT_ITERS
    else:
        raise ValueError(sweep)
    fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
    for value in values:
        row = keep(rows, **{sweep: value})[0]
        ax.plot(row["history"]["func_res"], label=f"{sweep}={value:g}")
    ax.set_yscale("log")
    ax.set_xlabel("Iteration k")
    ax.set_ylabel(r"$f(x_k)-f^\star$")
    ax.set_title(f"Subgradient {loss}, normalized, sweep over {sweep}")
    ax.legend()
    savefig(fig, f"subgradient_{loss}_{sweep}_sweep")
    return fig

for loss in SUBGRADIENT_LOSSES:
    fig = plot_subgradient_parameter_sweep(loss, "R")
    plt.show()
    fig = plot_subgradient_parameter_sweep(loss, "n_iters")
    plt.show()

## Report Notes

- Use semilog-y plots: the x-axis (`k`, time, matrix-vector products) is linear; residual and gradient/subgradient norm are logarithmic.
- A roughly straight residual curve on semilog axes supports linear/geometric convergence. A curve that flattens on semilog axes supports sublinear behavior.
- Compare slopes visually: steeper downward residual curves mean faster convergence.
- As `sigma` increases, convergence should slow. As `mu` increases, strongly convex smooth runs should generally improve.
- Subgradient residuals should be slower and less smooth than the smooth-method residuals; subgradient norms need not decrease monotonically.